<a href="https://colab.research.google.com/github/vkjadon/hugging_face/blob/main/hf_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Enable GPU Runtime

Navigate to Runtime → Change runtime type → T4 GPU → Save.



Verify GPU access:

In [ ]:
!nvidia-smi

## Install Libraries

The -q flag suppresses verbose output.

In [ ]:
!pip install -q transformers datasets huggingface_hub accelerate

## Authentication
For private models or pushing to Hub, authenticate with your token:

In [ ]:
from huggingface_hub import login
login()  # Opens interactive prompt

For a cleaner workflow, use Colab secrets:

To set up secrets, click the key icon in Colab's left sidebar and add HF_TOKEN with your Hugging Face access token.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

In [3]:
from datasets import load_dataset

dataset = load_dataset("imdb")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [4]:
train_data = dataset["train"]
example = train_data[0]
print(example)

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [ ]:
for example in train_data["label"]:
    print(example)

In [7]:
texts = train_data["text"]

print(texts[:5])

['I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, e

Load a Model Pipeline

In [ ]:
from transformers import pipeline

classifier = pipeline("text-classification", device=0)  # device=0 uses GPU
result = classifier("I love this course!")
print(result)

Load a Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("imdb", split="train[:100]")
print(dataset[0])

Check Device Placement


In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0)}")

## Memory Management
Colab's free tier provides approximately 15GB of GPU memory on the T4. Use these techniques to work within that limit.

In [ ]:
import torch
import gc

del model  # Delete the model variable
gc.collect()
torch.cuda.empty_cache()

##Load in Lower Precision

In [ ]:
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-hf",
    torch_dtype=torch.float16,
    device_map="auto"
)

##Mount Google Drive
Save models and checkpoints to persist across sessions:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Save model
model.save_pretrained('/content/drive/MyDrive/my_model')

Download Files Locally

In [ ]:
from google.colab import files
files.download('output.csv')

In [ ]:
# 2. Search for models
from huggingface_hub import list_models

models = list(list_models(pipeline_tag="text-classification", sort="downloads", limit=5))
for m in models:
    print(f"{m.id}: {m.downloads:,} downloads")

# 3. Try the top model
from transformers import pipeline

In [ ]:
classifier = pipeline("text-classification", model=models[0].id, device=0)
print(classifier("This is amazing!"))

In [ ]:
classifier = pipeline("text-classification", model=models[2].id, device=0)
print(classifier("This is amazing!"))